In [ ]:
import time
start_time = time.time()
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
import spark_sdk as ss
import yaml
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import col, sum as sum_, count, max as max_, datediff, lit, min as min_, desc, when, unix_timestamp, row_number, struct, collect_list,\
to_date, countDistinct, exp, lit
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import pytz

import spark_sdk as ss
ss.PySpark(yarn=False, num_executors=48, driver_memory='32g', executor_memory='32g',
           add_on_config0=('spark.port.maxRetries', '1000'),
           add_on_config1=("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog"),
           add_on_config2=('spark.databricks.delta.retentionDurationCheck.enabled', 'false'),
           add_on_config3=('spark.sql.sources.partitionOverwriteMode', 'dynamic')
          )
spark = ss.PySpark().spark

from pyspark.sql.functions import to_date

full_dataset = spark.read.format('parquet').load('file:/home/baond12/projects/AI_Recommend_System_2025/BTL_HTTM/CS-recommendation-system/backend/full_dataset.parquet')\
.withColumn('event_date', to_date('TimePlaced'))

# Lấy danh sách 7 ngày cuối cùng có trong dữ liệu
max_date_row = full_dataset.agg(max_("event_date")).collect()[0][0]
# Tính toán ngày bắt đầu validation (cách max date 6 ngày)
val_date = max_date_row - timedelta(days=6)

start_train_date = val_date - timedelta(days=7)

end_train_date = val_date - timedelta(days=1)

print(f"Start Training Date: {start_train_date} -- End Training Date: {end_train_date}")
print(f"Validation Date: {val_date}")

train_data_7days = full_dataset.filter(f"event_date <= '{end_train_date}' and event_date <= '{start_train_date}'")
train_data_1days = full_dataset.filter(f"event_date = '{end_train_date}'")

val_data = full_dataset.filter(f"event_date = '{val_date}'")

item_7days_metrics = train_data_7days.groupBy("ProductId").agg(
        count("ProductId").alias("nb_purchas_7days"),
        sum_('Price').alias("revenue_7days"))

item_1days_metrics = train_data_1days.groupBy("ProductId").agg(
        count("ProductId").alias("nb_purchase_1days"),
        sum_('Price').alias("revenue_1days"))

item_metrics = item_7days_metrics.join(item_1days_metrics, ['ProductId'], 'outer').fillna(0)

# item_metrics.show(2)

def normalize_score(df, score_col, group_cols=['ProductId']):
    stats = df.groupBy(*group_cols).agg(
        min_(score_col).alias("min_score"),
        max_(score_col).alias("max_score")
    )
    df = df.join(stats, on=group_cols, how="left")
    df = df.withColumn(
        f"{score_col}_norm",
        when(col("max_score") != col("min_score"),
             (col(score_col) - col("min_score")) / (col("max_score") - col("min_score"))
        ).otherwise(lit(0))
    ).drop("min_score", "max_score", score_col)
    df = df.withColumnRenamed(f"{score_col}_norm", score_col)
    return df

item_metrics_norm = normalize_score(item_metrics, 'nb_purchas_7days', group_cols=['ProductId'])
item_metrics_norm = normalize_score(item_metrics_norm, 'revenue_7days', group_cols=['ProductId'])
item_metrics_norm = normalize_score(item_metrics_norm, 'nb_purchase_1days', group_cols=['ProductId'])
item_metrics_norm = normalize_score(item_metrics_norm, 'revenue_1days', group_cols=['ProductId'])

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
from scipy.stats import norm

norm_cdf_udf = udf(lambda z: float(norm.cdf(z)), DoubleType())

item_metrics_norm = item_metrics_norm.withColumn("nb_purchas_7days_scale", norm_cdf_udf(col("nb_purchas_7days")))
item_metrics_norm = item_metrics_norm.withColumn("revenue_7days_scale", norm_cdf_udf(col("revenue_7days")))
item_metrics_norm = item_metrics_norm.withColumn("nb_purchase_1days_scale", norm_cdf_udf(col("nb_purchase_1days")))
item_metrics_norm = item_metrics_norm.withColumn("revenue_1days_scale", norm_cdf_udf(col("revenue_1days")))

item_metrics_norm = item_metrics_norm.withColumn(
    "final_score",
    0.2 * col("nb_purchas_7days_scale") + 0.3 * col("revenue_7days_scale") + 0.2 * col("nb_purchase_1days_scale") + 0.3 * col("revenue_1days_scale")
).select('ProductId', 'final_score')

item_metrics_norm = item_metrics_norm.withColumn('d', lit(val_date.strftime('%Y-%m-%d')))
item_metrics_norm.write.format('parquet')\
.mode('overwrite').option('overwriteSchema', 'true')\
.partitionBy(['d'])\
.save("file:/mnt/projects-data/recommendation/dev_folder/baond12/btl_httm/item_metrics.parquet")